In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_6_try_1.pickle

trying: ['time']
me:  0
trying: ['IPython']
me:  0
trying: ['flights_df']


me:  1
trying: ['pd']
me:  0
trying: ['orig_output']
me:  6
trying: ['sklearn']
me:  0
trying: ['np']
me:  0
trying: ['sp']
me:  0
trying: ['matplotlib']
me:  0
trying: ['plt']
me:  0
Declaring variable time
Declaring variable IPython
Declaring variable pd
Declaring variable sklearn
Declaring variable np
Declaring variable sp
Declaring variable matplotlib
Declaring variable plt
Declaring variable flights_df
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 7 ###

# Optimized for cudf
# 1) Filter only the `origin` column by the SEA mask
# 2) Use value_counts (GPU‐native) to get per-origin counts
# 3) Sum the small counts series (few entries) instead of scanning the full mask
sea_counts = flights_df['origin'][flights_df['dest'] == 'SEA'].value_counts()
sea_total  = sea_counts.sum()
(sea_counts['EWR'] / sea_total, sea_counts['JFK'] / sea_total)

CPU times: user 66.6 ms, sys: 24.6 ms, total: 91.2 ms
Wall time: 90.8 ms


(0.46673464185572267, 0.5332653581442773)

In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_7_try_4.pickle

migration speed (bps): 775637321.2404065
---------------------------
variables to migrate:
pd 72
sea_counts 48
flights_df 48
time 72
sea_total 32
sklearn 72
IPython 72
np 72
sp 72
matplotlib 72
orig_output 48
plt 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high/checkpoints/post_cell_7_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['flights_df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 6 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['flights_df']
Modified dataframes
======= Cell 7 =====

In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/opt_cell_exec_info_7_try_4.pkl",
    "wb",
) as f:
    pickle.dump(opt_cell_exec_info[7], f)

In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/annotated/checkpoints/post_cell_7.pickle

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output